In [1]:
import polars as pl
import numpy as np
import os
import io
import chess.pgn
import re
from typing import Dict, List, Any
from math import floor

In [2]:
INPUT = '../../../data/sample_data.pgn'

In [10]:
import re
import polars as pl

def parse_pgn_to_polars(pgn_file: str) -> pl.DataFrame:
    """
    Parse a PGN file and extract chess game metadata into a Polars DataFrame.
    
    Parameters
    ----------
    pgn_file : str
        Path to the PGN file.
    
    Returns
    -------
    pl.DataFrame
        DataFrame with one row per game, containing metadata and movetext.
    """
    # Read entire PGN file
    with open(pgn_file, "r", encoding="utf-8") as f:
        content = f.read()

    # Split games: they are separated by double newlines after result
    games = re.split(r"\n\n(?=\[Event )", content.strip())

    parsed_games: List[Dict[str, Any]] = []
    for game in games:
        # Extract all tags like [Key "Value"]
        tags = dict(re.findall(r'\[(\w+)\s+"([^"]+)"\]', game))

        # Extract movetext (everything after last ])
        moves_match = re.split(r'\]\s*\n', game, maxsplit=1)
        movetext = moves_match[1].strip() if len(moves_match) > 1 else ""

        # Store data
        parsed_games.append({**tags, "Moves": movetext})

    # Convert to Polars DataFrame
    df = pl.DataFrame(parsed_games)

    formats_to_keep = ["Rated Blitz game",
                       "Rated Bullet game",
                       "Rated Rapid game"]

    df = df.filter(pl.col('Event').str.contains('|'.join(formats_to_keep)))

    df = df.filter(pl.col('Termination') != 'Abandoned')

    df = df.with_columns(
    pl.when(pl.col("Event").str.contains("Rapid")).then(pl.lit("Rapid"))
     .when(pl.col("Event").str.contains("Bullet")).then(pl.lit("Bullet"))
     .when(pl.col("Event").str.contains("Blitz")).then(pl.lit("Blitz"))
     .otherwise(pl.col("Event"))
     .alias("TimeControl")
     )
    
    
    # Calculate number of moves for each game
    def count_moves(moves_str: str) -> int:
        # Find all move numbers followed by a dot (e.g., "1.", "2.", ...)
        move_numbers = re.findall(r'\d+\.', moves_str)
        # Each move number represents a pair (white and black), but the last move may be incomplete
        # Count the number of moves by counting move numbers, then check if there is an extra move at the end
        # Count all moves (white and black) by finding all SAN moves (excluding comments and clock info)
        moves = re.findall(r'\d+\.\s*([^{}]+)', moves_str)
        # Split moves by spaces and filter out empty strings and result (e.g., "1-0", "0-1", "1/2-1/2")
        san_moves = []
        for m in moves:
            san_moves += [x for x in re.split(r'\s+', m.strip()) if x and not re.match(r'^\d+\.\.\.$', x)]
        # Remove result at the end if present
        san_moves = [mv for mv in san_moves if not re.match(r'^(1-0|0-1|1/2-1/2)$', mv)]
        return len(san_moves)

    df = df.with_columns(
        pl.col("Moves").map_elements(count_moves).alias("NumMoves")
    )
    df = df.select(['TimeControl','WhiteElo', 'BlackElo',
                    'Termination', 'Moves','Result', 'NumMoves'])




    return df




df = parse_pgn_to_polars(INPUT)
df.head(3)

# 35, 27, 27


TimeControl,WhiteElo,BlackElo,Termination,Moves,Result,NumMoves
str,str,str,str,str,str,i64
"""Bullet""","""1706""","""1671""","""Time forfeit""","""[Site ""https://lichess.org/VsU…","""0-1""",139
"""Bullet""","""2262""","""2191""","""Normal""","""[Site ""https://lichess.org/dAP…","""1-0""",112
"""Bullet""","""2279""","""2339""","""Normal""","""[Site ""https://lichess.org/aoC…","""0-1""",124


In [ ]:
game = chess.pgn.read_game(io.StringIO(df['Moves'][0]))

In [ ]:
board = game.board()
for i, move in enumerate(game.mainline_moves()):
    if i == 20:
        break
    board.push(move)
board